# Vision Assist, Phase 3: COCO lane + ground hazards (kerbs, tactile paving)

Builds on Phase 2. Runtime > Change runtime type > **T4 GPU**. Run the cells in order.

1. **COCO lane** (2 minutes, no training): exports the standard YOLOv8n to `coco320.onnx`.
2. **Ground model** (about 1 hour): downloads public Roboflow Universe datasets, merges them into two classes
   (`kerb`, `tactile_paving`), trains YOLOv8n, exports `ground416.onnx`.

Send both files back. Nothing goes into the app until it has been tested on our own street footage.

In [ ]:
!pip -q install ultralytics onnx roboflow
from google.colab import drive
drive.mount('/content/drive')
import os; OUT = '/content/drive/MyDrive/vision_assist_phase3'; os.makedirs(OUT, exist_ok=True); print('Saving to', OUT)

## 1. COCO lane (no training)

In [ ]:
from ultralytics import YOLO
import shutil
YOLO('yolov8n.pt').export(format='onnx', imgsz=320, opset=12, simplify=True)
shutil.copy('yolov8n.onnx', f'{OUT}/coco320.onnx')
import onnxruntime as ort
s = ort.InferenceSession(f'{OUT}/coco320.onnx'); print('input', s.get_inputs()[0].shape, 'output', s.get_outputs()[0].shape)  # expect [1,84,2100]
from google.colab import files; files.download(f'{OUT}/coco320.onnx')

## 2. Ground hazards: pick the datasets

Get a free Roboflow API key: roboflow.com > Settings > API Keys. Open each dataset on Roboflow Universe first and check its
pictures look like UK pavements. Edit the list if you find better ones. Version `None` = the latest.

In [ ]:
from getpass import getpass
API_KEY = getpass('Roboflow API key: ')
DATASETS = [  # (workspace, project, version)
    ('work-rdjfc', 'tactile_pavement', None),
    ('thesis-jjmi5', 'tactile-paving-segmentation', None),
    ('sayan-o7dha', 'curbdetect-ogxre', None),
    ('stevens-wj2qk', 'identify-curb', None),
]
# Every source class is mapped by its name to one of ours. Names that match nothing are dropped.
CLASSES = ['kerb', 'tactile_paving']  # stairs are already in the Phase 2 model
def our_class(name):
    n = name.lower().replace('-', ' ').replace('_', ' ')
    if any(w in n for w in ('tactile', 'braille', 'blister', 'guide block', 'warning block')): return 1
    if any(w in n for w in ('curb', 'kerb')): return 0
    return None

In [ ]:
from roboflow import Roboflow
import yaml, glob, shutil, os
rf = Roboflow(api_key=API_KEY)
ROOT = '/content/ground'; shutil.rmtree(ROOT, ignore_errors=True)
for split in ('train', 'valid'):
    os.makedirs(f'{ROOT}/{split}/images', exist_ok=True); os.makedirs(f'{ROOT}/{split}/labels', exist_ok=True)
counts = {c: 0 for c in CLASSES}; used = []
for ws, pj, ver in DATASETS:
    try:
        proj = rf.workspace(ws).project(pj)
        v = proj.version(ver) if ver else max(proj.versions(), key=lambda x: int(str(x.version).split('/')[-1]))
        loc = f'/content/src/{ws}_{pj}'
        v.download('yolov8', location=loc, overwrite=True)
    except Exception as e:
        print('SKIPPED', ws, pj, '->', e); continue
    names = yaml.safe_load(open(f'{loc}/data.yaml'))['names']
    names = list(names.values()) if isinstance(names, dict) else names
    mapping = {i: our_class(n) for i, n in enumerate(names)}
    print(ws, pj, 'classes:', {n: (CLASSES[mapping[i]] if mapping[i] is not None else 'dropped') for i, n in enumerate(names)})
    n_img = 0
    for split_src in ('train', 'valid', 'test'):
        split = 'valid' if split_src != 'train' else 'train'
        for lab in glob.glob(f'{loc}/{split_src}/labels/*.txt'):
            out = []
            for line in open(lab).read().split('\n'):
                p = line.split()
                if len(p) < 5: continue
                c = mapping.get(int(p[0]))
                if c is None: continue
                vals = list(map(float, p[1:]))
                if len(vals) > 4:  # a polygon (segmentation dataset): turn it into its bounding box
                    xs, ys = vals[0::2], vals[1::2]
                    x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
                    vals = [(x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1]
                out.append(f'{c} ' + ' '.join(f'{x:.6f}' for x in vals[:4])); counts[CLASSES[c]] += 1
            if not out: continue
            stem = os.path.splitext(os.path.basename(lab))[0]
            imgs = glob.glob(f'{loc}/{split_src}/images/{stem}.*')
            if not imgs: continue
            tag = f'{ws}_{pj}_{stem}'
            shutil.copy(imgs[0], f'{ROOT}/{split}/images/{tag}{os.path.splitext(imgs[0])[1]}')
            open(f'{ROOT}/{split}/labels/{tag}.txt', 'w').write('\n'.join(out)); n_img += 1
    used.append((ws, pj, n_img))
yaml.safe_dump({'path': ROOT, 'train': 'train/images', 'val': 'valid/images', 'names': {i: c for i, c in enumerate(CLASSES)}}, open(f'{ROOT}/data.yaml', 'w'))
print('\nImages used per dataset:', used)
print('Boxes per class:', counts)
print('Train images:', len(os.listdir(f'{ROOT}/train/images')), ' Valid images:', len(os.listdir(f'{ROOT}/valid/images')))
low = [c for c, n in counts.items() if n < 150]
if low: print('\nWARNING: few examples for', low, '- this class will be unreliable. Add a dataset, or our own labelled photos, before trusting it.')

## 3. Train (about 1 hour on a T4). Saves to Drive, so a disconnect can resume.

In [ ]:
from ultralytics import YOLO
import os
last = f'{OUT}/runs/ground/weights/last.pt'
if os.path.exists(last):
    model = YOLO(last); model.train(resume=True)
else:
    model = YOLO('yolov8n.pt')
    model.train(data=f'{ROOT}/data.yaml', epochs=60, imgsz=416, batch=32, patience=15, project=f'{OUT}/runs', name='ground', exist_ok=True)

## 4. Check it, class by class

In [ ]:
best = f'{OUT}/runs/ground/weights/best.pt'
m = YOLO(best); r = m.val(data=f'{ROOT}/data.yaml', imgsz=416)
for i, c in enumerate(r.ap_class_index):
    print(f'{CLASSES[c]:15s} mAP50 {r.box.ap50[i]:.3f}   precision {r.box.p[i]:.3f}   recall {r.box.r[i]:.3f}')
print('all            mAP50', round(r.box.map50, 3))

## 5. Export for the app and download

In [ ]:
import shutil, onnxruntime as ort
p = YOLO(best).export(format='onnx', imgsz=416, opset=12, simplify=True)
shutil.copy(p, f'{OUT}/ground416.onnx')
s = ort.InferenceSession(f'{OUT}/ground416.onnx'); print('input', s.get_inputs()[0].shape, 'output', s.get_outputs()[0].shape)  # expect [1,6,3549]
from google.colab import files; files.download(f'{OUT}/ground416.onnx')